In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
def tsm_lin(q):
    return q / q.sum()

def tsm_sm(q, scale):
    return F.softmax(scale * 2 * q, dim=0)

def get_p_opt(y, alpha, n_iter=40):
    y = y.detach().double()
    alpha = torch.as_tensor(alpha, dtype=torch.float64, device=y.device).detach()

    log_y = torch.where(y > 0, torch.log(y), -torch.inf)

    # Solve p* = clamp(y, λ, exp(2α)λ), with sum(p*) = 1
    # using η = log(λ)
    log_n = torch.log(torch.tensor(y.numel(), dtype=torch.float64, device=y.device))

    lo = -log_n - 2 * alpha   # sum(p*) <= 1
    hi = -log_n               # sum(p*) >= 1

    for _ in range(n_iter):
        eta = (lo + hi) / 2

        log_p = torch.clamp(log_y, min=eta, max=eta + 2 * alpha)
        mass = torch.exp(log_p).sum()

        if mass < 1:
            lo = eta
        else:
            hi = eta

    eta = (lo + hi) / 2
    log_p_opt = torch.clamp(log_y, min=eta, max=eta + 2 * alpha)

    return torch.exp(log_p_opt)

def get_kl(y, p):
    return torch.sum(torch.special.xlogy(y, y / p))

In [ ]:
SCALE = 3

q = torch.tensor([1, 1, 0, 0], dtype=torch.float)
y = tsm_lin(q)

s = torch.tensor([0.9, 0.2, -.9, -1], dtype=torch.float)
z = SCALE * s
p = F.softmax(z, dim=0)

p_opt = get_p_opt(y, SCALE)


kl = get_kl(y, p)
E_s = get_kl(p_opt, p)
E_ir = get_kl(y, p_opt)
E_sr = torch.dot((y - p_opt), torch.log(p_opt / p))

print(kl)
print(E_s)
print(E_ir)
print(E_sr)